## Load the Data

In [1]:
import pandas as pd

In [2]:
# Load the data into a DataFrame
df = pd.read_json('all.jsonl', lines=True)

In [3]:
# Seperate each answer into its own row
human_answers = df["human_answers"].explode().reset_index(drop=True)
gpt_answers = df["chatgpt_answers"].explode().reset_index(drop=True)


In [4]:
# Combine into a single DataFrame and shuffle
combined_df = pd.DataFrame({
    "text": pd.concat([human_answers, gpt_answers], ignore_index=True),
    "label": [0] * len(human_answers) + [1] * len(gpt_answers)
})
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

Remember that human answers are labeled 0, and gpt answers are labeled 1!

In [5]:
# Split into training, validation, and test sets
num_samples = len(combined_df)
train = combined_df.iloc[:int(0.8 * num_samples)]
val = combined_df.iloc[int(0.8 * num_samples):int(0.9 * num_samples)]
test = combined_df.iloc[int(0.9 * num_samples):]

---
## Tokenize and Clean

In [7]:
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

In [8]:
# Check if GPU is available and set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [10]:
# Convert to Hugging Face Datasets and drop missing values
train_ds = Dataset.from_pandas(train.dropna())
val_ds = Dataset.from_pandas(val.dropna())
test_ds = Dataset.from_pandas(test.dropna())

In [11]:
# Load tokenizer and tokenize the data
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

def tokenize_function(samples):
    return tokenizer(
        samples['text'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

# Tokenize all datasets
train_tokenized = train_ds.map(tokenize_function, batched=True)
val_tokenized = val_ds.map(tokenize_function, batched=True)
test_tokenized = test_ds.map(tokenize_function, batched=True)

# Set format for PyTorch
train_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

Map: 100%|██████████| 8541/8541 [00:02<00:00, 3352.08 examples/s]


---
## Train roBERTa model

In [12]:
# Define model
model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=2
)

# Define training parameters
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
)

# Set up Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 3862.92it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [13]:
# Run training
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.592293,0.626064
2,0.590369,0.640204
3,0.049633,0.024355


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.18it/s]


TrainOutput(global_step=25638, training_loss=0.29455181442251577, metrics={'train_runtime': 15367.7561, 'train_samples_per_second': 13.345, 'train_steps_per_second': 1.668, 'total_flos': 5.395960456639488e+16, 'train_loss': 0.29455181442251577, 'epoch': 3.0})

In [14]:
# Evaluate on test set
results = trainer.evaluate(test_tokenized)
print("Evaluation results:")
print(results)

Training Loss,Validation Loss,Epoch
0.049633,0.032149,3


Evaluation results:
{'eval_loss': 0.03214895725250244}


---
## Evaluate Model

In [16]:
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import numpy as np

In [18]:
# Get predictions
predictions = trainer.predict(test_tokenized)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

In [ ]:
# F1 Score
f1 = f1_score(y_true, y_pred, average='weighted')
print(f"F1 Score (weighted): {f1:.4f}")
print(f"F1 Score (macro): {f1_score(y_true, y_pred, average='macro'):.4f}")

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:")
print(cm)

# Full classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Human (0)', 'LLM (1)']))

In [ ]:
# Visualize Confusion Matrix using sklearn's built-in display
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Human', 'LLM']).plot(
    ax=ax, 
    cmap='Blues',
    values_format='d'
)
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
from lime.lime_text import LimeTextExplainer

In [ ]:
# Create a prediction function for LIME
def predict_with_model(texts):
    """Predict probabilities for a list of texts."""
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
    
    return probs.cpu().numpy()

def predict_proba(texts):
    """Wrapper for LIME - takes list of strings, returns array of probabilities."""
    return predict_with_model(texts)

# Create LIME explainer
explainer = LimeTextExplainer(class_names=['Human', 'LLM'])
print("LIME explainer created!")

In [ ]:
# Example: Explain a single prediction
# Get a sample from test data
sample_text = test_tokenized[int(len(test_tokenized)/2)]['text']
sample_label = test_tokenized[int(len(test_tokenized)/2)]['label']

print(f"Sample text (first 200 chars): {sample_text[:200]}...")
print(f"True label: {'LLM' if sample_label == 1 else 'Human'}")

# Generate explanation
explanation = explainer.explain_instance(
    sample_text, 
    predict_proba, 
    num_features=10,
    num_samples=500
)

# Show top positive and negative features
print("\n=== LIME Explanation ===")
print(f"Prediction: {explanation.predict_proba}")
print(f"Predicted class: {'LLM' if explanation.predict() == 1 else 'Human'}")

print("\nTop features supporting prediction:")
for feature, weight in explanation.as_list():
    print(f"  {feature:30s} {weight:+.4f}")

In [ ]:
# Visualize LIME explanation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Feature importance bar chart
features = [f[0] for f in explanation.as_list()]
weights = [f[1] for f in explanation.as_list()]
colors = ['green' if w > 0 else 'red' for w in weights]

axes[0].barh(features, weights, color=colors)
axes[0].axvline(x=0, color='black', linestyle='-', linewidth=0.5)
axes[0].set_xlabel('Weight (supporting Human ← → supporting LLM)')
axes[0].set_title('LIME: Top Features for Prediction')

# Plot 2: Prediction probabilities
pred_proba = explanation.predict_proba
axes[1].bar(['Human', 'LLM'], pred_proba, color=['#3498db', '#e74c3c'])
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('Probability')
axes[1].set_title(f'Prediction: {"LLM" if explanation.predict() == 1 else "Human"}')

for i, v in enumerate(pred_proba):
    axes[1].text(i, v + 0.02, f'{v:.2%}', ha='center')

plt.tight_layout()
plt.show()

print(f"\n🔍 LIME Analysis:")
print(f"   Predicted: {'LLM' if explanation.predict() == 1 else 'Human'}")
print(f"   Confidence: {max(pred_proba):.2%}")